# Aşama 2b — NCM'yi BitNet gibi (ternary) eğitmek: %86'ya yaklaşıyor mu?

**Araştırma sorusu.** Eğitim sonrası ternary kuantalama (PTQ) NCM Sudoku-hard modelini %86.3'ten %0'a düşürdü
(Aşama 1). Aynı model **ternary ağırlıklarla eğitilirse** (QAT, BitNet b1.58 yöntemi) kalite ne kadar geri gelir?

**Deneyler** (hepsi aynı veri, aynı mimari, aynı değerlendirme):

| Ad | Başlangıç | Ağırlıklar | Ne ölçer |
|---|---|---|---|
| **E3** | Resmî FP checkpoint | Ternary (gövde), QAT ince ayar | En ucuz yol: FP modeli ternary'ye "alıştırmak" yetiyor mu? |
| **E1** | Rastgele | FP | Kendi eğitim döngümüzün kalitesi (kontrol). Aynı bütçede E2'nin referansı |
| **E2** | Rastgele | Ternary (gövde) | Asıl soru: baştan ternary eğitim (native QAT) |

**Önerilen sıra:** 1–6 (kontroller) → E3 → E1 ve E2. E3 en kısa süren ve tek başına en bilgilendirici deney.

**Neyi FP bırakıyoruz:** Giriş izdüşümü (`wte`) ve seviye çıkış katmanları (`lm_heads`), yani lineer parametrelerin
yaklaşık %0.9'u. BitNet'in gömme/çıkış katmanlarını FP tutma pratiğiyle uyumlu; PTQ teşhisinde giriş izdüşümü
(`wte`) en hassas katman çıktı; çıkış katmanları tek tek ternary yapılınca zararsız (≈%85). Ternary katmanlarda aktivasyonlar token başına INT8'e kuantalanır.

**Varsayımlarımız (makalede Sudoku için verilmeyen ayarlar):** koşullandırma bozma ortalama 0.5 (seviyeler boyunca
0.75'ten 0.25'e doğrusal azalan), kayıp ağırlığı α = 0. E1 bu varsayımların makul olup olmadığını gösterir.

**Çalışma zamanı:** GPU (A100 önerilir; T4'te çalışır ama çok yavaştır). Checkpoint'ler Drive'a yazılır; oturum koparsa aynı
hücreyi tekrar çalıştırınca kaldığı adımdan devam eder.

## 1. GPU, Drive ve yollar

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/ncm_ptq'
QAT_DIR = f'{OUT_DIR}/qat'
os.makedirs(QAT_DIR, exist_ok=True)
DATA_NPZ = f'{OUT_DIR}/sudoku_hard_sflm_seed42.npz'    # Aşama 2a çıktısı
assert os.path.exists(DATA_NPZ), 'Önce Aşama 2a notebook\'unu çalıştır'

## 2. Resmî repo (Aşama 0/1 ile aynı commit), checkpoint ve test verisi

In [ ]:
%cd /content
!test -d ncm || git clone https://github.com/HazyResearch/ncm.git
%cd /content/ncm
!git checkout -q c70274273f489abc73832b2b98584a8483b6b203 && git rev-parse HEAD
!pip install -q omegaconf hydra-core "transformers>=4.51" regex
!pip install -q -U "huggingface_hub[cli]"
!pip list 2>/dev/null | grep -iE "^(torch|transformers|numpy|omegaconf|hydra-core|huggingface.hub) " | tee /content/drive/MyDrive/ncm_ptq/env_versions.txt
!hf download hazyresearch/ncm-checkpoints --include "sudoku/hard/*" --local-dir artifacts
!hf download hazyresearch/ncm-datasets --repo-type dataset --include "sudoku/hard/*" --local-dir data

## 3. Eğitim modülünü yaz (`ncm_qat.py`)

In [ ]:
%%writefile /content/ncm_qat.py
"""
NCM Sudoku — Aşama 2b: resmî NCM kodu üzerinde FP ve ternary (BitNet b1.58) eğitim döngüsü.

Resmî repoda (HazyResearch/ncm) eğitim kodu yok; bu modül makaledeki Algoritma 1'i ve resmî
`generate()` / `quant.py` mantığını izleyerek teacher-forcing eğitimini kurar.

Girdi (resmî `generate()` ile aynı yapı, tek ileri geçişte tüm seviyeler):
    x = [ önek gömmesi (P=91) ; seviye-0 sorgusu (l0=1) ; idx_to_ncm_input(kodlar) (Σ l_k, k≥1) ]
    - önek gömmesi  = tokenizer.encoder.wte(önek token'ları)       (config: raw_token_prefix=true)
    - seviye-0      = net.l0_queries                                (config: query_level0=true)
    - seviye k≥1    = önceki seviyelerin birikmiş rekonstrüksiyonu, l_k uzunluğuna 'area' ile küçültülmüş
Kayıp (makale Denk. 10):  L = − Σ_k w_k Σ_i log p(r_k^i | z̃^(k−1)),   w_k = l_k^(−α) / Σ_j l_j^(1−α)
Koşullandırma bozma (makale §3.4, Ek F.3): koşul kodlarının bir kısmı aynı seviyenin kod-kitabından
    düzgün rastgele kodlarla değiştirilir; oran seviyeler boyunca doğrusal azalır. Hedefler temiz kalır.
    NOT: Sudoku için kesin oranlar ve α makalede verilmiyor; varsayılanlar (ortalama 0.5, α=0) bizim
    seçimimizdir ve FP kontrol deneyi (resmî %86'ya yaklaşılabiliyor mu?) ile sınanır.

Ternary: BitNet b1.58 — ağırlık absmean ile {−1,0,+1}·γ, aktivasyon token başına absmax INT8,
    her ikisi de STE ile. Bias ve normalizasyon katmanları tam hassasiyette kalır.
    Varsayılan kapsam 'body' (transformer blokları); giriş (wte) ve seviye çıkış katmanları (lm_heads)
    FP kalır (BitNet'in gömme/çıkış katmanlarını FP tutma pratiği; PTQ taramasında bu katmanlar en hassas çıktı).
"""

from __future__ import annotations

import json
import math
import os
import pickle
import sys
import time
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# 1. Ternary katman (BitNet b1.58) — repo bağımsız, yerelde birim testli
# ---------------------------------------------------------------------------

def ste_ternary_absmean(w: torch.Tensor, eps: float = 1e-5, lam: float = 1.0, per_channel: bool = False) -> torch.Tensor:
    """lam=1: tam ternary (BitNet b1.58). 0<lam<1: kademeli geçiş W + lam·(W_t − W) (HF 1.58-bit tarifi).
    per_channel=True: her çıkış satırına ayrı γ (W ≈ diag(γ)·T); çıkarımda γ çıktıya bir kez uygulanır, çarpmasızlık korunur."""
    gamma = (w.abs().mean(dim=1, keepdim=True) if per_channel else w.abs().mean()).clamp(min=eps)
    wq = (w / gamma).round().clamp(-1, 1) * gamma
    return w + lam * (wq - w).detach()


def ste_int8_absmax_per_token(x: torch.Tensor, eps: float = 1e-5, lam: float = 1.0) -> torch.Tensor:
    scale = x.abs().amax(dim=-1, keepdim=True).clamp(min=eps) / 127.0
    xq = (x / scale).round().clamp(-127, 127) * scale
    return x + lam * (xq - x).detach()


class TernaryLinear(nn.Linear):
    """nn.Linear ile aynı arayüz; ileri geçişte ternary ağırlık + (isteğe bağlı) INT8 aktivasyon, STE ile."""

    def __init__(self, in_features, out_features, bias=True, act_int8=True, per_channel=False, device=None, dtype=None):
        super().__init__(in_features, out_features, bias=bias, device=device, dtype=dtype)
        self.act_int8 = act_int8
        self.per_channel = per_channel
        self.quant_lambda = 1.0   # 1.0 = tam kuantalı; yalnızca eğitimin başındaki kademeli geçişte <1

    def forward(self, x):
        lam = self.quant_lambda
        w = ste_ternary_absmean(self.weight.float(), lam=lam, per_channel=self.per_channel).to(x.dtype)
        if self.act_int8:
            x = ste_int8_absmax_per_token(x, lam=lam)
        return F.linear(x, w, self.bias.to(x.dtype) if self.bias is not None else None)

    @torch.no_grad()
    def ternary_stats(self) -> Dict[str, float]:
        g = (self.weight.abs().mean(dim=1, keepdim=True) if self.per_channel else self.weight.abs().mean()).clamp(min=1e-5)
        t = (self.weight / g).round().clamp(-1, 1)
        n = t.numel()
        return {"neg": float((t == -1).sum()) / n, "zero": float((t == 0).sum()) / n, "pos": float((t == 1).sum()) / n}


def _is_io(name: str) -> bool:
    return ("wte" in name) or ("lm_head" in name) or ("ctx_" in name)


SCOPES = {
    "body": lambda n: not _is_io(n),
    "all": lambda n: True,
}


def ternarize_linears(net: nn.Module, scope: str = "body", act_int8: bool = True, per_channel: bool = False) -> List[str]:
    """Kapsamdaki nn.Linear katmanlarını aynı ağırlıklarla TernaryLinear'a çevirir (yerinde). Değişen adları döndürür."""
    sel = SCOPES[scope]
    targets = [(n, m) for n, m in net.named_modules()
               if type(m) is nn.Linear and sel(n)]
    for name, lin in targets:
        parent_name, _, attr = name.rpartition(".")
        parent = net.get_submodule(parent_name) if parent_name else net
        new = TernaryLinear(lin.in_features, lin.out_features, bias=lin.bias is not None, act_int8=act_int8,
                            per_channel=per_channel, device=lin.weight.device, dtype=lin.weight.dtype)
        with torch.no_grad():
            new.weight.copy_(lin.weight)
            if lin.bias is not None:
                new.bias.copy_(lin.bias)
        setattr(parent, attr, new)
    return [n for n, _ in targets]


# ---------------------------------------------------------------------------
# 2. Kayıp ağırlıkları ve koşullandırma bozma — repo bağımsız, yerelde birim testli
# ---------------------------------------------------------------------------

def set_quant_lambda(net: nn.Module, lam: float) -> int:
    """Tüm TernaryLinear katmanlarında kademeli kuantalama katsayısını ayarlar; kaç katman etkilendiğini döndürür."""
    n = 0
    for m in net.modules():
        if isinstance(m, TernaryLinear):
            m.quant_lambda = float(lam); n += 1
    return n


def level_position_weights(levels: List[int], alpha: float) -> torch.Tensor:
    """Makale Denk. 10: her kod pozisyonu için w_k = l_k^(−α) / Σ_j l_j^(1−α). Toplamları her zaman 1'dir."""
    denom = sum(l ** (1.0 - alpha) for l in levels)
    return torch.cat([torch.full((l,), (l ** (-alpha)) / denom) for l in levels])


def linear_corruption_rates(n_cond_levels: int, mean: float = 0.5, spread: float = 0.5) -> List[float]:
    """Koşul olarak kullanılan seviyeler (0..K−2) için doğrusal azalan oranlar; ortalama = mean.
    spread = ilk ve son seviye oranları arasındaki fark."""
    if n_cond_levels == 1:
        return [mean]
    hi, lo = mean + spread / 2, mean - spread / 2
    assert 0.0 <= lo and hi <= 1.0, "oranlar [0,1] dışında"
    return [hi + (lo - hi) * k / (n_cond_levels - 1) for k in range(n_cond_levels)]


def corrupt_codes_random_rate(idx: torch.Tensor, levels: List[int], vocab_sizes: List[int],
                              low: float = 0.0, high: float = 1.0, skip_level0: bool = True,
                              generator: Optional[torch.Generator] = None) -> torch.Tensor:
    """
    Resmî LM1B config'indeki `code_corruption` bloğunun yorumu (mode: uniform, random_rate: true, low, high,
    skip_level0). Bu bloğu okuyan kod yayımlanmadığı için yorum bizimdir:
      - her dizi için bir oran r ~ U(low, high) çekilir (ortalama (low+high)/2; low=0, high=1 → 0.5),
      - koşul olarak kullanılan seviyelerdeki (0..K−2) her kod r olasılıkla aynı seviyenin kod-kitabından düzgün
        rastgele bir kodla değiştirilir,
      - skip_level0=True ise en kaba seviye (seviye 0) hiç bozulmaz; son seviye zaten koşul olarak kullanılmaz.
    """
    B = idx.size(0)
    r = low + (high - low) * torch.rand(B, 1, device=idx.device, generator=generator)
    out = idx.clone()
    off = 0
    for k, l in enumerate(levels):
        if k < len(levels) - 1 and not (skip_level0 and k == 0):
            seg = out[:, off:off + l]
            mask = torch.rand(seg.shape, device=idx.device, generator=generator) < r
            rnd = torch.randint(0, vocab_sizes[k], seg.shape, device=idx.device, generator=generator)
            out[:, off:off + l] = torch.where(mask, rnd, seg)
        off += l
    return out


def corrupt_codes(idx: torch.Tensor, levels: List[int], vocab_sizes: List[int], rates: List[float],
                  generator: Optional[torch.Generator] = None) -> torch.Tensor:
    """idx: (B, Σ l_k). Seviye k<K−1 kodlarını rates[k] olasılıkla aynı seviyenin kod-kitabından düzgün rastgele
    kodlarla değiştirir. Son seviye hiçbir zaman koşul olarak kullanılmadığından değiştirilmez."""
    out = idx.clone()
    off = 0
    for k, l in enumerate(levels):
        if k < len(levels) - 1 and rates[k] > 0:
            seg = out[:, off:off + l]
            mask = torch.rand(seg.shape, device=idx.device, generator=generator) < rates[k]
            rnd = torch.randint(0, vocab_sizes[k], seg.shape, device=idx.device, generator=generator)
            out[:, off:off + l] = torch.where(mask, rnd, seg)
        off += l
    return out


# ---------------------------------------------------------------------------
# 3. Resmî kodla yükleme, girdi kurma, değerlendirme
# ---------------------------------------------------------------------------

def load_official(ncm_root: str, config_path: str, ckpt_dir: str, device: str = "cuda",
                  load_generator: bool = True):
    """Resmî tokenizer'ı (dondurulmuş) ve üreteci yükler. load_generator=False ise üreteç rastgele başlatılır."""
    if ncm_root not in sys.path:
        sys.path.insert(0, ncm_root)
    from omegaconf import OmegaConf
    from models.next_concept import NextConceptTransformer
    from tokenization.vqvae_tokenizer import VQVAETokenizer

    cfg = OmegaConf.load(config_path)
    cfg.tokenizer.vqvae.checkpoint_path = os.path.join(ckpt_dir, cfg.checkpoints.tokenizer)
    tk = VQVAETokenizer.from_config(cfg.tokenizer)
    tk.vqvae = tk.vqvae.to(device).eval()
    for p in tk.vqvae.parameters():
        p.requires_grad_(False)
    net = NextConceptTransformer(vqvae_local=tk.vqvae, config=cfg.model).to(device)
    if load_generator:
        sd = torch.load(os.path.join(ckpt_dir, cfg.checkpoints.generator), map_location="cpu", weights_only=False)
        net.load_state_dict(sd["model"], strict=True)
    else:
        # Resmî sınıf kendi _init_weights'ini (nanoGPT tarzı N(0, 0.02)) __init__ içinde çağırmıyor; yayımlanmamış
        # eğitim betiğinin çağırdığını varsayıyoruz. Aksi hâlde nn.Embedding varsayılanı N(0,1) olurdu.
        # Bu, sınıfın __init__'te açıkça başlattığı parametreleri (l0_queries, wpe) etkilemez.
        net.apply(net._init_weights)
    assert bool(cfg.model.get("raw_token_prefix", False)) and not bool(cfg.model.get("contextual_prefix", False)), \
        "Bu modül raw_token_prefix=true, contextual_prefix=false kurulumunu varsayar (Sudoku config'i)"
    assert bool(cfg.model.get("query_level0", False)), "Bu modül query_level0=true varsayar"
    assert not bool(getattr(net, "predict_level0", False)), "predict_level0 desteklenmiyor"
    return cfg, tk, net


def build_inputs(net, tk, seqs: torch.Tensor, rates=None,
                 generator: Optional[torch.Generator] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    seqs: (B, 180) = [BOS] bulmaca(89) [BOS] çözüm(89).  Döndürür: x (B, P+Σl, D), hedef kodlar idx (B, Σl).
    Önek ve kod girdileri dondurulmuş tokenizer'dan gelir (gradyansız); yalnızca l0_queries eğitilebilir.
    rates: None (bozma yok) | seviye başına oran listesi (doğrusal mod) |
           {"mode": "random_uniform", "low", "high", "skip_level0"} (LM1B tarzı mod)
    """
    vq = tk.vqvae.vq
    P = int(net.prefix_len)
    prefix_tokens, solution = seqs[:, :P], seqs[:, P:]
    with torch.no_grad():
        idx = tk.vqvae.encode(solution)                       # (B, Σl)
        if rates is None:
            cond_idx = idx
        elif isinstance(rates, dict):
            cond_idx = corrupt_codes_random_rate(idx, list(vq.levels), list(vq.vqvae_vocab_sizes),
                                                 rates["low"], rates["high"], rates["skip_level0"], generator)
        else:
            cond_idx = corrupt_codes(idx, list(vq.levels), list(vq.vqvae_vocab_sizes), rates, generator)
        x_cond = vq.idx_to_ncm_input(cond_idx)                # (B, Σ_{k≥1} l_k, D)
        z_prefix = tk.vqvae.encoder.wte(prefix_tokens)[:, :P, :]
    q0 = net.build_l0_query_input(seqs.size(0), z_prefix.dtype)  # (B, l0, D), gradyanlı
    return torch.cat([z_prefix, q0, x_cond], dim=1), idx


def ncm_loss(net, x: torch.Tensor, idx: torch.Tensor, pos_w: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Ağırlıklı kod çapraz entropisi (dizi başına toplam, batch ortalaması) ve logitler."""
    logits = net(x)                                           # (B, Σl, V)
    ce = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), idx.reshape(-1), reduction="none")
    loss = (ce.view(idx.shape) * pos_w.to(ce.device)[None, :]).sum(dim=1).mean()
    return loss, logits


def kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, pos_w: torch.Tensor,
            temperature: float = 1.0) -> torch.Tensor:
    """Konum başına KL(p_öğretmen || p_öğrenci)·T², CE ile aynı seviye ağırlıklarıyla; dizi başına toplam, batch ort."""
    T = float(temperature)
    log_ps = F.log_softmax(student_logits.float() / T, dim=-1)
    log_pt = F.log_softmax(teacher_logits.float() / T, dim=-1)
    kl = (log_pt.exp() * (log_pt - log_ps)).sum(-1)          # (B, Σl)
    return (kl * pos_w.to(kl.device)[None, :]).sum(dim=1).mean() * (T * T)


def teacher_inputs(teacher, student_x: torch.Tensor, P: int) -> torch.Tensor:
    """Öğrencinin girdisinde yalnızca öğrenilen l0 sorgularını öğretmeninkiyle değiştirir (önek ve kodlar ortak)."""
    q0_t = teacher.build_l0_query_input(student_x.size(0), student_x.dtype)
    l0 = q0_t.size(1)
    return torch.cat([student_x[:, :P], q0_t, student_x[:, P + l0:]], dim=1).detach()


@torch.no_grad()
def per_level_accuracy(logits: torch.Tensor, idx: torch.Tensor, levels: List[int]) -> List[float]:
    pred = logits.argmax(-1)
    out, off = [], 0
    for l in levels:
        out.append(float((pred[:, off:off + l] == idx[:, off:off + l]).float().mean()))
        off += l
    return out


@torch.no_grad()
def consistency_check(net, tk, prefixes: torch.Tensor) -> Dict[str, float]:
    """
    Girdi kurucumuzun resmî generate() ile birebir uyumlu olduğunu sınar:
    greedy generate → kodlar g;  g'den teacher-forcing girdisi kur → forward → argmax  ==  g  olmalı
    (her seviyenin tahmini yalnızca önceki seviyelerin kodlarına bağlı olduğu için).
    """
    net.eval()
    g = net.generate(None, prefixes, temperature=0.0, top_k=0, top_p=1.0, cfg_scale=0.0, cfg_null_eos=None)
    vq = tk.vqvae.vq
    P = int(net.prefix_len)
    x_cond = vq.idx_to_ncm_input(g)
    z_prefix = tk.vqvae.encoder.wte(prefixes)[:, :P, :]
    x = torch.cat([z_prefix, net.build_l0_query_input(prefixes.size(0), z_prefix.dtype), x_cond], dim=1)
    pred = net(x).argmax(-1)
    agree = (pred == g).float()
    off, per = 0, []
    for l in vq.levels:
        per.append(float(agree[:, off:off + l].mean()))
        off += l
    return {"agree_all": float(agree.mean()), "agree_per_level": per,
            "exact_sequences": float((pred == g).all(dim=1).float().mean())}


def _decode(tk, ids):
    eos, pad = tk.pre_tokenizer.eos_token_id, tk.pre_tokenizer.pad_token_id
    ids = list(ids)
    if eos in ids:
        ids = ids[:ids.index(eos)]
    return tk.pre_tokenizer.decode([t for t in ids if t not in (eos, pad)], skip_special_tokens=True)


def score_sudoku(hyps: List[str], refs: List[str]) -> Dict[str, float]:
    """Resmî scripts/eval_merge.py::exact() ile aynı skorlama."""
    def cells(t):
        return t.replace("|", "")

    def valid(t):
        g = cells(t)
        if len(g) != 81 or not g.isdigit() or "0" in g:
            return False
        rows_ = [g[i * 9:(i + 1) * 9] for i in range(9)]
        cols_ = ["".join(r[j] for r in rows_) for j in range(9)]
        boxes = ["".join(rows_[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3))
                 for br in (0, 3, 6) for bc in (0, 3, 6)]
        return all(sorted(x) == list("123456789") for x in rows_ + cols_ + boxes)

    n = len(hyps)
    return {"solved_pct": 100.0 * sum(h == r for h, r in zip(hyps, refs)) / n,
            "cell_pct": 100.0 * sum(sum(a == b for a, b in zip(cells(h), cells(r))) / max(1, len(cells(r)))
                                    for h, r in zip(hyps, refs)) / n,
            "valid_pct": 100.0 * sum(valid(h) for h in hyps) / n, "n": n}


@torch.no_grad()
def evaluate_test(net, tk, items: List, refs: List, batch_size: int = 128, device: str = "cuda",
                  amp_dtype: Optional[torch.dtype] = None) -> Dict[str, float]:
    """Resmî eval_shard.py (Sudoku, greedy) ile aynı üretim + resmî skorlama."""
    net.eval()
    P = int(net.prefix_len)
    pad = tk.pre_tokenizer.pad_token_id
    ref_txt = [r if isinstance(r, str) else _decode(tk, r) for r in refs]
    hyps = []
    for s in range(0, len(items), batch_size):
        x = torch.tensor([(list(it)[:P] + [pad] * P)[:P] for it in items[s:s + batch_size]], device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None and device == "cuda"):
            g = net.generate(None, x, temperature=0.0, top_k=0, top_p=1.0, cfg_scale=0.0, cfg_null_eos=None)
            _, zl = tk.vqvae.vq.idx_to_zhats(g, return_zlevels=True)
            ids = tk.vqvae.decoder(zl[-1]).argmax(-1).tolist()
        hyps += [_decode(tk, r) for r in ids]
    return score_sudoku(hyps, ref_txt)


# ---------------------------------------------------------------------------
# 4. Eğitim döngüsü (makale Tablo 6: AdamW β=(0.9,0.99), wd 0.01, tepe lr 3e-4, taban 3e-5, ısınma, clip 1)
# ---------------------------------------------------------------------------

@dataclass
class TrainConfig:
    run_name: str
    steps: int = 5000
    batch_size: int = 256
    micro_batch: int = 128            # bellek yetmezse küçült; batch_size'ın böleni olmalı
    peak_lr: float = 3e-4
    floor_lr: float = 3e-5
    warmup: int = 500
    weight_decay: float = 0.01
    betas: Tuple[float, float] = (0.9, 0.99)
    grad_clip: float = 1.0
    alpha: float = 0.0
    corruption_mode: str = "linear"   # "linear" (seviye başına sabit, doğrusal azalan) | "random_uniform" (LM1B tarzı)
    corruption_mean: float = 0.5      # yalnızca linear modda
    corruption_spread: float = 0.5    # yalnızca linear modda
    corruption_low: float = 0.0       # yalnızca random_uniform modda
    corruption_high: float = 1.0      # yalnızca random_uniform modda
    corruption_skip_level0: bool = True  # yalnızca random_uniform modda
    seed: int = 0
    log_every: int = 50
    eval_every: int = 1000            # tutulan eğitim dilimi üzerinde teacher-forcing kaybı/doğruluğu
    ckpt_every: int = 500
    test_n_during: int = 256          # ara test değerlendirmesinde kaç test bulmacası (0 = kapalı)
    quant_warmup: int = 0             # >0: ilk bu kadar adımda kuantalama katsayısı λ 0→1 doğrusal artar
    kd_weight: float = 0.0            # >0: kayıp = (1−w)·CE + w·KD; train(teacher=...) gerekir
    kd_temperature: float = 1.0


def lr_at(step: int, cfg: TrainConfig) -> float:
    if step < cfg.warmup:
        return cfg.peak_lr * (step + 1) / cfg.warmup
    prog = min(1.0, (step - cfg.warmup) / max(1, cfg.steps - cfg.warmup))
    return cfg.floor_lr + 0.5 * (cfg.peak_lr - cfg.floor_lr) * (1 + math.cos(math.pi * prog))


def make_optimizer(net: nn.Module, cfg: TrainConfig):
    decay, no_decay = [], []
    for n, p in net.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.dim() >= 2 else no_decay).append(p)
    return torch.optim.AdamW([{"params": decay, "weight_decay": cfg.weight_decay},
                              {"params": no_decay, "weight_decay": 0.0}],
                             lr=cfg.peak_lr, betas=cfg.betas)


def pick_amp_dtype() -> Optional[torch.dtype]:
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def train(net, tk, train_seqs: np.ndarray, holdout_seqs: np.ndarray, cfg: TrainConfig, out_dir: str,
          test_items=None, test_refs=None, device: str = "cuda", teacher=None) -> List[Dict]:
    """Drive'a düzenli checkpoint yazar; aynı run_name ile yeniden çağrılırsa kaldığı adımdan devam eder.
    teacher: kd_weight>0 ise dondurulmuş FP öğretmen (aynı tokenizer'la yüklenmiş NextConceptTransformer)."""
    if cfg.kd_weight > 0:
        assert teacher is not None, "kd_weight>0 için teacher verilmeli"
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    P = int(net.prefix_len)
    use_lambda = cfg.quant_warmup > 0
    os.makedirs(out_dir, exist_ok=True)
    ckpt_path = os.path.join(out_dir, f"{cfg.run_name}.pt")
    log_path = os.path.join(out_dir, f"{cfg.run_name}.log.json")
    vq = tk.vqvae.vq
    levels = list(vq.levels)
    pos_w = level_position_weights(levels, cfg.alpha).to(device)
    if cfg.corruption_mode == "random_uniform":
        rates = {"mode": "random_uniform", "low": cfg.corruption_low, "high": cfg.corruption_high,
                 "skip_level0": cfg.corruption_skip_level0}
    elif cfg.corruption_mode == "linear":
        rates = linear_corruption_rates(len(levels) - 1, cfg.corruption_mean, cfg.corruption_spread) \
            if cfg.corruption_mean > 0 else None
    else:
        raise ValueError(f"bilinmeyen corruption_mode: {cfg.corruption_mode}")
    amp = pick_amp_dtype()
    scaler = torch.cuda.amp.GradScaler(enabled=(amp == torch.float16))
    opt = make_optimizer(net, cfg)
    step, logs = 0, []
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        net.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); step = ck["step"]
        if "scaler" in ck:
            scaler.load_state_dict(ck["scaler"])
        logs = ck.get("logs", [])
        print(f"[devam] {cfg.run_name}: adım {step}'ten sürüyor")

    rng = np.random.default_rng(cfg.seed + step)
    gen = torch.Generator(device=device); gen.manual_seed(cfg.seed * 1000003 + step)
    hold = torch.tensor(holdout_seqs[:1024], dtype=torch.long, device=device)
    assert cfg.batch_size % cfg.micro_batch == 0
    n_micro = cfg.batch_size // cfg.micro_batch
    t0 = time.time()
    net.train()
    while step < cfg.steps:
        for g_ in opt.param_groups:
            g_["lr"] = lr_at(step, cfg)
        lam = min(1.0, (step + 1) / cfg.quant_warmup) if use_lambda else 1.0
        if use_lambda:
            set_quant_lambda(net, lam)
        opt.zero_grad(set_to_none=True)
        bidx = rng.integers(0, len(train_seqs), size=cfg.batch_size)
        batch = torch.tensor(train_seqs[bidx], dtype=torch.long, device=device)
        tot, tot_ce, tot_kd = 0.0, 0.0, 0.0
        for m in range(n_micro):
            mb = batch[m * cfg.micro_batch:(m + 1) * cfg.micro_batch]
            x, idx = build_inputs(net, tk, mb, rates, gen)
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                ce, s_logits = ncm_loss(net, x, idx, pos_w)
                if cfg.kd_weight > 0:
                    with torch.no_grad():
                        t_logits = teacher(teacher_inputs(teacher, x, P))
                    kd = kd_loss(s_logits, t_logits, pos_w, cfg.kd_temperature)
                    loss = (1.0 - cfg.kd_weight) * ce + cfg.kd_weight * kd
                    tot_kd += float(kd) / n_micro
                else:
                    loss = ce
            scaler.scale(loss / n_micro).backward()
            tot += float(loss) / n_micro
            tot_ce += float(ce) / n_micro
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
        scaler.step(opt); scaler.update()
        step += 1

        if step % cfg.log_every == 0:
            rec = {"step": step, "loss": round(tot, 5), "lr": lr_at(step - 1, cfg), "grad_norm": float(gn),
                   "elapsed_min": round((time.time() - t0) / 60, 2)}
            extra = ""
            if cfg.kd_weight > 0:
                rec["ce"] = round(tot_ce, 5); rec["kd"] = round(tot_kd, 5)
                extra += f"  ce={tot_ce:.4f}  kd={tot_kd:.4f}"
            if use_lambda:
                rec["quant_lambda"] = round(lam, 4)
                extra += f"  λ={lam:.2f}"
            logs.append(rec)
            print(f"[{cfg.run_name}] adım {step:6d}/{cfg.steps}  kayıp={tot:.4f}{extra}  lr={rec['lr']:.2e}  "
                  f"|g|={float(gn):.2f}  ({rec['elapsed_min']:.1f} dk)", flush=True)

        if step % cfg.eval_every == 0 or step == cfg.steps:
            net.eval()
            if use_lambda:
                set_quant_lambda(net, 1.0)   # ara ölçümler her zaman gerçek ternary modelle
            with torch.no_grad(), torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                xs, ids = build_inputs(net, tk, hold[:512], None)
                hl, hlog = ncm_loss(net, xs, ids, pos_w)
            rec = {"step": step, "holdout_tf_loss": float(hl),
                   "holdout_tf_acc_per_level": [round(a, 4) for a in per_level_accuracy(hlog, ids, levels)]}
            if cfg.test_n_during and test_items is not None:
                # Üretim FP32'de yapılır (resmî değerlendirmeyle aynı); yalnızca eğitim karma hassasiyetlidir.
                rec["test_subset"] = evaluate_test(net, tk, test_items[:cfg.test_n_during],
                                                   test_refs[:cfg.test_n_during], device=device, amp_dtype=None)
            logs.append(rec)
            print(f"  [eval] {json.dumps(rec)}", flush=True)
            if use_lambda:
                set_quant_lambda(net, lam)
            net.train()

        if step % cfg.ckpt_every == 0 or step == cfg.steps:
            ck = {"model": net.state_dict(), "opt": opt.state_dict(), "step": step, "logs": logs,
                  "cfg": asdict(cfg)}
            if scaler.is_enabled():
                ck["scaler"] = scaler.state_dict()
            torch.save(ck, ckpt_path + ".tmp"); os.replace(ckpt_path + ".tmp", ckpt_path)
            json.dump(logs, open(log_path, "w"), indent=1)
    if use_lambda:
        set_quant_lambda(net, 1.0)
    net.eval()
    return logs

## 4. Veriyi yükle

Önemli: S-FLM'nin doğrulama kümesi NCM'nin **test** kümesinin ta kendisi (Aşama 2a'da 2000/2000 doğrulandı).
Bu yüzden eğitim sırasında izleme için doğrulama kümesini değil, eğitim kümesinden ayırdığımız 1000 bulmacayı
(`holdout`) kullanıyoruz. Test kümesine yalnızca ara alt-küme kontrolü (ilk 256) ve en sondaki tam değerlendirme
için bakılır; hiperparametre seçimi test kümesine bakılarak yapılmaz.

In [ ]:
import sys, json, pickle, importlib, numpy as np, torch
sys.path.insert(0, '/content')
import ncm_qat as Q; importlib.reload(Q)

d = np.load(DATA_NPZ)
meta = json.loads(str(d['meta']))
print(meta)
assert meta['ncm_test_in_train'] == 0 and meta['ncm_test_in_valid'] == 2000
all_train = d['train'].astype(np.int64)
train_seqs, holdout_seqs = all_train[:-1000], all_train[-1000:]
test_items = pickle.load(open('/content/ncm/data/sudoku/hard/puzzles.pickle', 'rb'))
test_refs = pickle.load(open('/content/ncm/data/sudoku/hard/solutions.pickle', 'rb'))
print('eğitim', train_seqs.shape, '| holdout', holdout_seqs.shape, '| test', len(test_items))

NCM = dict(ncm_root='/content/ncm', config_path='/content/ncm/config/eval/sudoku-hard.yaml',
           ckpt_dir='/content/ncm/artifacts/sudoku/hard')

## 5. Kontroller (eğitimden önce mutlaka)

1. **Girdi kurucusu = resmî `generate()`**: resmî modelle greedy üret, aynı kodlardan teacher-forcing girdisi kur,
   tekrar tahmin et. `agree_all` ≈ 1.0 olmalı (küçük sayısal farklar dışında birebir).
2. **Resmî modelin kendi verisindeki teacher-forcing kaybı ve seviye doğrulukları**: düşük kayıp, yüksek doğruluk
   beklenir. Bu, verinin ve kod çıkarımının (tokenizer.encode) modelin eğitildiği veriyle uyumlu olduğunu gösterir.
3. **Değerlendiricimiz = resmî sonuç**: tam testte `solved_pct` = 86.30 olmalı.

In [ ]:
cfg_m, tk, net = Q.load_official(**NCM)
dev = 'cuda'
pre = torch.tensor([list(it)[:91] for it in test_items[:256]], device=dev)
print('Kontrol 1 (girdi kurucusu):', Q.consistency_check(net, tk, pre))

levels = list(tk.vqvae.vq.levels)
pos_w = Q.level_position_weights(levels, 0.0).to(dev)
with torch.no_grad():
    hb = torch.tensor(holdout_seqs[:512], device=dev)
    x, idx = Q.build_inputs(net, tk, hb, None)
    net.eval(); l, lg = Q.ncm_loss(net, x, idx, pos_w)
print('Kontrol 2 (resmî model, holdout TF kaybı):', round(float(l), 4))
print('          seviye doğrulukları:', [round(a, 3) for a in Q.per_level_accuracy(lg, idx, levels)])

r_off = Q.evaluate_test(net, tk, test_items, test_refs)
print('Kontrol 3 (resmî model, tam test):', r_off)
json.dump(r_off, open(f'{QAT_DIR}/official_fp_test.json', 'w'), indent=1)

## 6. Bütçe ayarı ve hız ölçümü

Makalenin bütçesi (50.000 adım × 1024 batch) Colab'da pratik değil. Aşağıdaki varsayılanlar kısa bir karşılaştırma
içindir; önemli olan **aynı bütçede E1 ile E2 arasındaki fark** ve E3'ün PTQ'ya (%0) göre ne kadar geri kazandığı.
Hücre 20 adım ölçüp her deneyin tahmini süresini yazar; gerekirse `STEPS_*` değerlerini buna göre ayarla.
`MICRO` bellek yetmezse (CUDA out of memory) küçültülür; sonucu değiştirmez.

In [ ]:
STEPS_FT, STEPS_SCRATCH = 1500, 4000
BATCH = 256
# MICRO GPU belleğine göre otomatik seçilir (A100 40/80 GB → 256, L4 24 GB → 128, T4 16 GB → 64).
# Bellek hatası (CUDA out of memory) alırsan aşağıdaki satırı elle küçült, örn. MICRO = 64.
_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
MICRO = 256 if _gb >= 35 else (128 if _gb >= 20 else 64)
print(f'GPU: {torch.cuda.get_device_name(0)} ({_gb:.0f} GB)  →  BATCH={BATCH}, MICRO={MICRO}')

import time, shutil
shutil.rmtree('/content/_probe', ignore_errors=True)   # eski ölçüm checkpoint'i varsa sil (yoksa devam eder, süre 0 çıkar)
probe =Q.TrainConfig(run_name='_hiz_olcumu', steps=20, batch_size=BATCH, micro_batch=MICRO, warmup=5,
                      log_every=10, eval_every=10**9, ckpt_every=10**9, test_n_during=0)
_, _, tmp = Q.load_official(**NCM)
Q.ternarize_linears(tmp, 'body')
t = time.time(); Q.train(tmp, tk, train_seqs, holdout_seqs, probe, '/content/_probe'); dt = (time.time() - t) / 20
print(f'≈ {dt:.2f} s/adım  →  E3 ≈ {dt*STEPS_FT/60:.0f} dk,  E1 ≈ E2 ≈ {dt*STEPS_SCRATCH/60:.0f} dk (her biri)')
del tmp; torch.cuda.empty_cache()

## 7. E3 — Resmî FP checkpoint'ten ternary QAT ince ayarı

Önce ternary'ye çevrilmiş ama henüz eğitilmemiş modelin sonucu ölçülür (PTQ ile aynı, ≈%0 beklenir), sonra ince ayar.

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM)
changed = Q.ternarize_linears(net, 'body')
print(len(changed), 'katman ternary')
r0 = Q.evaluate_test(net, tk, test_items[:256], test_refs[:256])
print('İnce ayar öncesi (ilk 256 test):', r0)

cfg_e3 = Q.TrainConfig(run_name='E3_ternary_ft', steps=STEPS_FT, batch_size=BATCH, micro_batch=MICRO,
                       peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=250, ckpt_every=250)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e3, QAT_DIR, test_items, test_refs)
r_e3 = Q.evaluate_test(net, tk, test_items, test_refs)
print('E3 tam test:', r_e3)
json.dump({'before_ft_256': r0, 'test': r_e3, 'cfg': cfg_e3.__dict__}, open(f'{QAT_DIR}/E3_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 8. E1 — FP, baştan eğitim (kontrol)

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM, load_generator=False)
cfg_e1 = Q.TrainConfig(run_name='E1_fp_scratch', steps=STEPS_SCRATCH, batch_size=BATCH, micro_batch=MICRO)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e1, QAT_DIR, test_items, test_refs)
r_e1 = Q.evaluate_test(net, tk, test_items, test_refs)
print('E1 tam test:', r_e1)
json.dump({'test': r_e1, 'cfg': cfg_e1.__dict__}, open(f'{QAT_DIR}/E1_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9. E2 — Ternary, baştan eğitim (native QAT)

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM, load_generator=False)
Q.ternarize_linears(net, 'body')
cfg_e2 = Q.TrainConfig(run_name='E2_ternary_scratch', steps=STEPS_SCRATCH, batch_size=BATCH, micro_batch=MICRO)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e2, QAT_DIR, test_items, test_refs)
r_e2 = Q.evaluate_test(net, tk, test_items, test_refs)
print('E2 tam test:', r_e2)
tern = {n: m.ternary_stats() for n, m in net.named_modules() if isinstance(m, Q.TernaryLinear)}
zero = np.mean([v['zero'] for v in tern.values()])
print(f'ortalama sıfır oranı: {zero:.3f}')
json.dump({'test': r_e2, 'mean_zero_frac': float(zero), 'cfg': cfg_e2.__dict__}, open(f'{QAT_DIR}/E2_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9b. Uzun bütçe — E1 ve E2'yi 3 kat daha uzun eğit (fark kapanıyor mu?)

4.000 adımda ternary ile FP arasındaki holdout kaybı farkı 0.053'ten 0.033'e daraldı. Bu bölüm aynı iki deneyi
`LONG_STEPS` (varsayılan 12.000) adımla, **yeni adlarla** sıfırdan çalıştırır; 4.000 adımlık sonuçlara dokunmaz.
Diğer bütün ayarlar E1/E2 ile aynıdır (aynı öğrenme oranı eğrisi, yalnızca daha uzun).

A100'de tahmini süre: E1-uzun ≈ 1 saat 50 dk, E2-uzun ≈ 2 saat 10 dk. **İki ayrı Colab oturumunda paralel**
çalıştırılabilir: ikisinde de 1–6. adımları çalıştır, sonra birinde yalnızca E1-uzun, diğerinde yalnızca E2-uzun
hücresini çalıştır. Oturum koparsa aynı adımlarla kaldığı yerden devam eder (checkpoint 500 adımda bir).

In [ ]:
LONG_STEPS = 12000
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM, load_generator=False)
cfg_e1l = Q.TrainConfig(run_name=f'E1_fp_scratch_{LONG_STEPS//1000}k', steps=LONG_STEPS, batch_size=BATCH,
                        micro_batch=MICRO, eval_every=2000, ckpt_every=500)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e1l, QAT_DIR, test_items, test_refs)
r_e1l = Q.evaluate_test(net, tk, test_items, test_refs)
print('E1-uzun tam test:', r_e1l)
json.dump({'test': r_e1l, 'cfg': cfg_e1l.__dict__}, open(f'{QAT_DIR}/E1_long_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

In [ ]:
LONG_STEPS = 12000
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM, load_generator=False)
Q.ternarize_linears(net, 'body')
cfg_e2l = Q.TrainConfig(run_name=f'E2_ternary_scratch_{LONG_STEPS//1000}k', steps=LONG_STEPS, batch_size=BATCH,
                        micro_batch=MICRO, eval_every=2000, ckpt_every=500)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e2l, QAT_DIR, test_items, test_refs)
r_e2l = Q.evaluate_test(net, tk, test_items, test_refs)
print('E2-uzun tam test:', r_e2l)
tern = {n: m.ternary_stats() for n, m in net.named_modules() if isinstance(m, Q.TernaryLinear)}
zero = np.mean([v['zero'] for v in tern.values()])
json.dump({'test': r_e2l, 'mean_zero_frac': float(zero), 'cfg': cfg_e2l.__dict__},
          open(f'{QAT_DIR}/E2_long_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9c. Teşhis — E1-uzun ezberliyor mu? (eğitimsiz, ≈5 dk)

E1-uzun'un son checkpoint'iyle aynı sayıda **eğitim** bulmacasını ve **test** bulmacasını çözdürür.
- Eğitimde çok yüksek, testte düşükse → ezber (aşırı öğrenme); daha güçlü düzenlileştirme (bozma) gerekir.
- İkisi de düşükse → ezber değil; tarif ya da kapasite sorunu.

In [ ]:
_, tk, net = Q.load_official(**NCM, load_generator=False)
ck = torch.load(f'{QAT_DIR}/E1_fp_scratch_12k.pt', map_location='cpu', weights_only=False)
net.load_state_dict(ck['model']); print('checkpoint adımı:', ck['step'])
N_DIAG = 500
tr_items = [list(map(int, s[:91])) for s in train_seqs[:N_DIAG]]
tr_refs = [list(map(int, s[91:])) for s in train_seqs[:N_DIAG]]
r_tr = Q.evaluate_test(net, tk, tr_items, tr_refs)
r_te = Q.evaluate_test(net, tk, test_items[:N_DIAG], test_refs[:N_DIAG])
print(f'EĞİTİM bulmacaları: {r_tr}')
print(f'TEST bulmacaları  : {r_te}')
json.dump({'checkpoint': 'E1_fp_scratch_12k', 'step': ck['step'], 'train': r_tr, 'test': r_te},
          open(f'{QAT_DIR}/diag_E1_long_train_vs_test.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9d. E1-LM1B-bozma — resmî LM1B config'indeki bozma ayarıyla FP, baştan, 12.000 adım

Resmî `config/eval/lm1b-*.yaml` dosyalarında, okuyan kodu yayımlanmamış bir blok var:
`code_corruption: {mode: uniform, random_rate: true, low: 0.0, high: 1.0, skip_level0: true}`
(bkz. HazyResearch/ncm issue #24). Bizim yorumumuz: her dizi için oran r ~ U(0, 1) çekilir ve koşul kodları bu
oranla rastgele kodlarla değiştirilir; en kaba seviye hiç bozulmaz.

**Tek değişiklik bozma tarifi.** Bütçe, öğrenme oranı eğrisi ve ısınma (500) E1-uzun ile aynı tutuldu ki fark
yalnızca bozmadan kaynaklansın. Karşılaştırma: E1-uzun (doğrusal 0.75→0.25, seviye 0 dahil) → %25.65.
Bakılacaklar: holdout kaybı 6.000. adımdan sonra yükselmeyi bırakıyor mu? Çözme oranı belirgin artıyor mu?
A100'de ≈1 saat 50 dk.

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM, load_generator=False)
cfg_e1r = Q.TrainConfig(run_name='E1_fp_scratch_12k_lm1bcorr', steps=12000, batch_size=BATCH, micro_batch=MICRO,
                        eval_every=2000, ckpt_every=500,
                        corruption_mode='random_uniform', corruption_low=0.0, corruption_high=1.0,
                        corruption_skip_level0=True)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_e1r, QAT_DIR, test_items, test_refs)
r_e1r = Q.evaluate_test(net, tk, test_items, test_refs)
print('E1-LM1B-bozma tam test:', r_e1r)
json.dump({'test': r_e1r, 'cfg': cfg_e1r.__dict__}, open(f'{QAT_DIR}/E1_lm1bcorr_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9e. E3-KD — FP öğretmenden damıtma + kademeli kuantalama (5.000 adım)

E3-uzun (5.000 adım, yalnızca CE) ile aynı tarif; iki fark: kayıp = 0.5·CE + 0.5·KL(öğretmen‖öğrenci), öğretmen resmî
FP NCM (dondurulmuş), ve ilk 500 adımda kuantalama katsayısı λ 0→1 (HF 1.58-bit tarifi). Ara ölçümler λ=1 ile yapılır.
Karşılaştırma: E3-uzun tam test %71.60.

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM)
Q.ternarize_linears(net, 'body')
_, _, teacher = Q.load_official(**NCM)          # dondurulmuş FP öğretmen (%86)
cfg_kd = Q.TrainConfig(run_name='E3_ternary_ft_5k_kd', steps=5000, batch_size=BATCH, micro_batch=MICRO,
                       peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500,
                       kd_weight=0.5, kd_temperature=1.0, quant_warmup=500)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_kd, QAT_DIR, test_items, test_refs, teacher=teacher)
r_kd = Q.evaluate_test(net, tk, test_items, test_refs)
print(f"E3-KD tam test: %{r_kd['solved_pct']:.2f}   (E3-uzun aynı adım: %71.60 | FP öğretmen: %86.15)")
json.dump({'test': r_kd, 'cfg': cfg_kd.__dict__}, open(f'{QAT_DIR}/E3_5k_kd_result.json', 'w'), indent=1)
del net, teacher; torch.cuda.empty_cache()

## 9f. E3-α1 — en kaba seviyeye daha çok ağırlık (α=1, her seviye eşit pay)

α=0'da tek kodluk en kaba seviyenin kayıptaki payı 1/322 ≈ %0,3; α=1'de her seviye 1/12 ≈ %8,3 alır (makale Denk. 10).
Hipotez: E3'ün takıldığı yer en kaba seviye (holdout TF doğruluğu ≈%81; testte en kaba kod doğruluğu E3 %76,7, FP %85,5). Diğer her şey 9e ile aynı; USE_KD=False ise damıtma
kapalı ve doğrudan E3-uzun (%71,60) ile karşılaştırılır.

In [ ]:
USE_KD = True    # 9e (E3-KD) işe yaradıysa True bırak; yaramadıysa False yap
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM)
Q.ternarize_linears(net, 'body')
teacher = Q.load_official(**NCM)[2] if USE_KD else None
run = 'E3_ternary_ft_5k_kd_a1' if USE_KD else 'E3_ternary_ft_5k_a1'
cfg_a1 = Q.TrainConfig(run_name=run, steps=5000, batch_size=BATCH, micro_batch=MICRO,
                       peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500, alpha=1.0,
                       kd_weight=0.5 if USE_KD else 0.0, kd_temperature=1.0, quant_warmup=500 if USE_KD else 0)
w = Q.level_position_weights(list(tk.vqvae.vq.levels), 1.0)
print('en kaba seviyenin kayıptaki payı:', round(float(w[0]), 4))
Q.train(net, tk, train_seqs, holdout_seqs, cfg_a1, QAT_DIR, test_items, test_refs, teacher=teacher)
r_a1 = Q.evaluate_test(net, tk, test_items, test_refs)
print(f"{run} tam test: %{r_a1['solved_pct']:.2f}   (E3-uzun: %71.60 | FP: %86.15)")
json.dump({'test': r_a1, 'cfg': cfg_a1.__dict__}, open(f'{QAT_DIR}/{run}_result.json', 'w'), indent=1)
del net, teacher; torch.cuda.empty_cache()

## 9g. E3-A16 — ternary ağırlık, INT8 aktivasyon KAPALI (W1.58A16)

E3-uzun ile birebir aynı tarif (α=0, damıtma yok, 5.000 adım); tek fark act_int8=False. Amaç: en kaba seviyedeki
açığın (testte en kaba kod doğruluğu E3 %76,7, FP %85,5) ternary ağırlıktan mı INT8 aktivasyondan mı geldiğini ayırmak. Karşılaştırma: %71.60.

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM)
changed = Q.ternarize_linears(net, 'body', act_int8=False)
assert all(not m.act_int8 for m in net.modules() if isinstance(m, Q.TernaryLinear))
print(len(changed), 'katman ternary, INT8 aktivasyon kapalı')
cfg_a16 = Q.TrainConfig(run_name='E3_ternary_ft_5k_a16', steps=5000, batch_size=BATCH, micro_batch=MICRO,
                        peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_a16, QAT_DIR, test_items, test_refs)
r_a16 = Q.evaluate_test(net, tk, test_items, test_refs)
print(f"E3-A16 tam test: %{r_a16['solved_pct']:.2f}   (E3-uzun, INT8 aktivasyonlu: %71.60 | FP: %86.15)")
json.dump({'test': r_a16, 'cfg': cfg_a16.__dict__, 'act_int8': False},
          open(f'{QAT_DIR}/E3_ternary_ft_5k_a16_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 9h. E3-PC — kanal (satır) başına ternary ölçek, W ≈ diag(γ)·T

E3-uzun ile aynı tarif (α=0, INT8 aktivasyon açık, 5.000 adım); tek fark her çıkış satırına ayrı γ. Ek bellek
matris başına satır sayısı kadar sayı (≈%0,2); çıkarımda γ çıktıya bir kez uygulanır. Karşılaştırma: %71.60.

In [ ]:
torch.manual_seed(0)
_, tk, net = Q.load_official(**NCM)
changed = Q.ternarize_linears(net, 'body', act_int8=True, per_channel=True)
assert all(m.per_channel and m.act_int8 for m in net.modules() if isinstance(m, Q.TernaryLinear))
print(len(changed), 'katman ternary, kanal başına ölçek, INT8 aktivasyon açık')
cfg_pc = Q.TrainConfig(run_name='E3_ternary_ft_5k_pc', steps=5000, batch_size=BATCH, micro_batch=MICRO,
                       peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500)
Q.train(net, tk, train_seqs, holdout_seqs, cfg_pc, QAT_DIR, test_items, test_refs)
r_pc = Q.evaluate_test(net, tk, test_items, test_refs)
print(f"E3-PC tam test: %{r_pc['solved_pct']:.2f}   (E3-uzun, tensör başına ölçek: %71.60 | FP: %86.15)")
json.dump({'test': r_pc, 'cfg': cfg_pc.__dict__, 'per_channel': True, 'act_int8': True},
          open(f'{QAT_DIR}/E3_ternary_ft_5k_pc_result.json', 'w'), indent=1)
del net; torch.cuda.empty_cache()

## 11. PTQ çöküş mekanizması (eğitim yok, yalnızca ölçüm, ≈20–30 dk)

**Gerekenler:** yalnızca 1–4. bölümler. Sonra bu bölümün iki hücresini sırayla çalıştır.

Üç soru:
- **A. Hata büyüklüğü mü, ternary'nin kendisi mi?** Gövdeye INT8/4/3, ternary ve *aynı büyüklükte rastgele Gauss
  gürültüsü* eklenir (ilk 500 test). Gürültü de aynı noktada çökertiyorsa sorun ternary değil, hatanın büyüklüğü.
- **B. Hangi katman/blok?** Her katman tek başına, her blok tek başına, ilk k ve son k blok ternary (ilk 256 test).
- **C. Hata seviyeler boyunca birikiyor mu?** Teacher-forcing doğruluğu ile üretim doğruluğu seviye seviye; FP'nin
  ürettiği koddan ilk ayrıldığı seviye; blok başına gizli durum sapması.

Sonuçlar `qat/ptq_mechanism.json` dosyasına adım adım yazılır; oturum koparsa hücre kaldığı yerden devam eder.

In [ ]:
%%writefile /content/ncm_diag.py
"""
PTQ çöküş mekanizması teşhisi (yalnızca ölçüm, eğitim yok).

Sorular:
  1. Hata büyüklüğü mü, ternary'nin kendisi mi?  -> Gauss gürültüsü ile hata eğrisi (rel_err ekseninde).
  2. Hangi katman / blok çökertiyor?             -> tek katman, tek blok ve kümülatif blok taramaları.
  3. Hata seviyeler boyunca birikiyor mu?        -> teacher-forcing (TF) ile üretim doğruluğu seviye seviye,
                                                    FP'nin üretiminden ilk ayrıldığı seviye, blok başına sapma.
Ağırlıklar yerinde değiştirilir ve her ölçümden sonra anlık görüntüden geri yüklenir.
"""
from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import ncm_qat as Q

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# --------------------------------------------------------------------------- ağırlık işlemleri

def linears(net: nn.Module):
    return [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear)]


def snapshot(net: nn.Module) -> Dict[str, torch.Tensor]:
    return {n: m.weight.detach().clone() for n, m in linears(net)}


@torch.no_grad()
def restore(net: nn.Module, snap: Dict[str, torch.Tensor]) -> None:
    for n, m in linears(net):
        m.weight.copy_(snap[n])


def _q_int_ch(w, bits):
    qmax = 2 ** (bits - 1) - 1
    s = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
    return (w / s).round().clamp(-qmax, qmax) * s


def _q_tern(w, per_channel=False):
    g = (w.abs().mean(dim=1, keepdim=True) if per_channel else w.abs().mean()).clamp(min=1e-8)
    return (w / g).round().clamp(-1, 1) * g


@torch.no_grad()
def perturb(net: nn.Module, names: List[str], kind: str, rel: float = 0.0, seed: int = 0) -> Dict[str, float]:
    """kind: tern | tern_ch | int8 | int4 | int3 | int2 | noise (Gauss, katman başına ||ΔW||/||W|| = rel)."""
    sel = set(names)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    err2 = ref2 = 0.0
    for n, m in linears(net):
        if n not in sel:
            continue
        w = m.weight.float()
        if kind == "tern":
            wq = _q_tern(w)
        elif kind == "tern_ch":
            wq = _q_tern(w, per_channel=True)
        elif kind.startswith("int"):
            wq = _q_int_ch(w, int(kind[3:]))
        elif kind == "noise":
            z = torch.randn(w.shape, generator=gen).to(w.device)
            wq = w + z * (rel * w.norm() / z.norm())
        else:
            raise ValueError(kind)
        err2 += float((wq - w).pow(2).sum()); ref2 += float(w.pow(2).sum())
        m.weight.copy_(wq.to(m.weight.dtype))
    return {"rel_err": (err2 / ref2) ** 0.5 if ref2 else 0.0, "n_layers": len(sel)}


# --------------------------------------------------------------------------- ölçümler

def _lvl_mean(a: torch.Tensor, levels: List[int]) -> List[float]:
    out, off = [], 0
    for l in levels:
        out.append(round(float(a[:, off:off + l].float().mean()), 4)); off += l
    return out


def _first_level(mask: torch.Tensor, levels: List[int]) -> torch.Tensor:
    """mask (B, Σl) bool: her örnek için mask'ın ilk True olduğu seviye; hiç yoksa len(levels)."""
    lvl_of_pos = torch.cat([torch.full((l,), i) for i, l in enumerate(levels)]).to(mask.device)
    big = torch.full_like(lvl_of_pos, len(levels))
    return torch.where(mask, lvl_of_pos[None, :], big[None, :]).min(dim=1).values


@torch.no_grad()
def generate_codes(net, tk, items, bs: int = 128, device: str = None):
    device = device or DEVICE
    """Q.evaluate_test ile aynı üretim; kodları ve çözülen metinleri döndürür."""
    net.eval()
    P = int(net.prefix_len)
    pad = tk.pre_tokenizer.pad_token_id
    codes, hyps = [], []
    for s in range(0, len(items), bs):
        x = torch.tensor([(list(it)[:P] + [pad] * P)[:P] for it in items[s:s + bs]], device=device)
        g = net.generate(None, x, temperature=0.0, top_k=0, top_p=1.0, cfg_scale=0.0, cfg_null_eos=None)
        _, zl = tk.vqvae.vq.idx_to_zhats(g, return_zlevels=True)
        ids = tk.vqvae.decoder(zl[-1]).argmax(-1).tolist()
        codes.append(g); hyps += [Q._decode(tk, r) for r in ids]
    return torch.cat(codes), hyps


@torch.no_grad()
def tf_logits(net, tk, seqs: np.ndarray, bs: int = 128, device: str = None):
    device = device or DEVICE
    net.eval()
    lg, ix = [], []
    for s in range(0, len(seqs), bs):
        x, idx = Q.build_inputs(net, tk, torch.as_tensor(seqs[s:s + bs], device=device), None)
        lg.append(net(x).float()); ix.append(idx)
    return torch.cat(lg), torch.cat(ix)


@torch.no_grad()
def block_outputs(net, tk, seqs: np.ndarray, device: str = None) -> List[torch.Tensor]:
    device = device or DEVICE
    outs = []
    hooks = [b.register_forward_hook(lambda _m, _i, o: outs.append((o[0] if isinstance(o, tuple) else o).float()))
             for b in net.transformer.blocks]
    try:
        x, _ = Q.build_inputs(net, tk, torch.as_tensor(seqs, device=device), None)
        net.eval(); net(x)
    finally:
        for h in hooks:
            h.remove()
    return outs


@torch.no_grad()
def fp_reference(net, tk, items, refs, seqs, levels, n_drift: int = 64) -> Dict:
    codes, hyps = generate_codes(net, tk, items)
    lg, idx = tf_logits(net, tk, seqs)
    return {"codes": codes, "logits": lg, "idx": idx, "blocks": block_outputs(net, tk, seqs[:n_drift]),
            "score": Q.score_sudoku(hyps, list(refs) if isinstance(refs[0], str) else [Q._decode(tk, r) for r in refs]),
            "n_drift": n_drift}


@torch.no_grad()
def measure(net, tk, items, refs, seqs, levels, fp: Optional[Dict] = None, full: bool = True) -> Dict:
    """full=False: yalnızca çözülen % (taramalar için hızlı). full=True: seviye ve blok düzeyinde teşhis."""
    ref_txt = [r if isinstance(r, str) else Q._decode(tk, r) for r in refs]
    codes, hyps = generate_codes(net, tk, items)
    sc = Q.score_sudoku(hyps, ref_txt)
    out = {"solved_pct": round(sc["solved_pct"], 2), "cell_pct": round(sc["cell_pct"], 2), "n": sc["n"]}
    if fp is not None:
        fl = _first_level(codes != fp["codes"][:len(codes)], levels)
        out["same_codes_as_fp_pct"] = round(100 * float((fl == len(levels)).float().mean()), 2)
        if not full:
            out["first_div_level_hist"] = torch.bincount(fl, minlength=len(levels) + 1).tolist()
    if not full:
        return out
    lg, idx = tf_logits(net, tk, seqs)
    out["tf_acc_per_level"] = _lvl_mean(lg.argmax(-1) == idx, levels)
    out["gen_acc_per_level"] = _lvl_mean(codes == idx, levels)          # üretilen kod = gerçek çözümün kodu
    ce = F.cross_entropy(lg.reshape(-1, lg.size(-1)), idx.reshape(-1), reduction="none").view(idx.shape)
    out["tf_ce_per_level"] = _lvl_mean(ce, levels)
    if fp is not None:
        lp, lq = F.log_softmax(fp["logits"], -1), F.log_softmax(lg, -1)
        out["kl_fp_q_per_level"] = _lvl_mean((lp.exp() * (lp - lq)).sum(-1), levels)
        out["first_div_level_hist"] = torch.bincount(fl, minlength=len(levels) + 1).tolist()
        bo = block_outputs(net, tk, seqs[:fp["n_drift"]])
        out["block_rel_drift"] = [round(float((a - b).norm() / b.norm()), 4) for a, b in zip(bo, fp["blocks"])]
    return out

In [ ]:
import re, pandas as pd
import ncm_diag as D; importlib.reload(D)
valid = np.load(DATA_NPZ)['valid'].astype(np.int64)        # = NCM test kümesi, aynı sıra (Aşama 2a)
_, tk, net = Q.load_official(**NCM); net.eval()
levels = list(tk.vqvae.vq.levels)
OUT = f'{QAT_DIR}/ptq_mechanism.json'
R = json.load(open(OUT)) if os.path.exists(OUT) else {}
def save(): json.dump(R, open(OUT, 'w'), indent=1)
snap = D.snapshot(net)
names = [n for n, _ in D.linears(net)]
body = [n for n in names if not Q._is_io(n)]; io = [n for n in names if Q._is_io(n)]
bidx = lambda n: int(re.search(r'blocks\.(\d+)\.', n).group(1))
assert all(re.search(r'blocks\.(\d+)\.', n) for n in body), [n for n in body if not re.search(r'blocks\.', n)]
nblk = 1 + max(bidx(n) for n in body)
print(len(body), 'gövde katmanı,', nblk, 'blok,', len(io), 'giriş/çıkış katmanı')

N, NS = 500, 256
fp = D.fp_reference(net, tk, test_items[:N], test_refs[:N], valid[:N], levels)
print('FP (ilk 500):', fp['score']['solved_pct'])

# A. hata eğrisi (gövde, ilk 500, ayrıntılı)
A = [('fp', None, 0), ('int8', 'int8', 0), ('int4', 'int4', 0), ('int3', 'int3', 0),
     ('tern', 'tern', 0), ('tern_ch', 'tern_ch', 0)] + \
    [(f'noise_{r}', 'noise', r) for r in (0.05, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5)]
for name, kind, rel in A:
    key = f'A/{name}'
    if key in R: continue
    info = D.perturb(net, body, kind, rel) if kind else {'rel_err': 0.0, 'n_layers': 0}
    R[key] = {**info, **D.measure(net, tk, test_items[:N], test_refs[:N], valid[:N], levels, fp)}
    D.restore(net, snap); save()
    print(f"{key:16s} rel_err={R[key]['rel_err']:.3f}  çözülen={R[key]['solved_pct']:6.2f}  FP ile aynı kod={R[key]['same_codes_as_fp_pct']:6.2f}")

# B. ternary tarama (ilk 256, hızlı)
G = {f'B/layer/{n}': [n] for n in names}
G.update({f'B/block/{i}': [n for n in body if bidx(n) == i] for i in range(nblk)})
G.update({f'B/first_k/{k}': [n for n in body if bidx(n) < k] for k in range(1, nblk)})
G.update({f'B/last_k/{k}': [n for n in body if bidx(n) >= nblk - k] for k in range(1, nblk)})
G['B/io_only'] = io
for i, (key, ns) in enumerate(G.items()):
    if key in R: continue
    info = D.perturb(net, ns, 'tern')
    R[key] = {**info, **D.measure(net, tk, test_items[:NS], test_refs[:NS], valid[:NS], levels, fp, full=False)}
    D.restore(net, snap); save()
    if i % 10 == 0: print(f'tarama {i+1}/{len(G)}')
print('bitti →', OUT)

In [ ]:
# Özet: bu hücrenin çıktısının tamamını kopyalayıp gönder
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
print('=== A. Hata eğrisi (gövde 32 katman, ilk 500 test) ===')
print(pd.DataFrame([{'ad': k[2:], 'rel_err': round(v['rel_err'], 3), 'çözülen%': v['solved_pct'],
                     'hücre%': v['cell_pct'], 'FP_ile_aynı_kod%': v['same_codes_as_fp_pct']}
                    for k, v in R.items() if k.startswith('A/')]).to_string(index=False))
for k in ['A/fp', 'A/int4', 'A/int3', 'A/tern', 'A/noise_0.2', 'A/noise_0.3']:
    if k in R:
        v = R[k]
        print(f"\n--- {k[2:]} ---")
        for f in ['tf_acc_per_level', 'gen_acc_per_level', 'kl_fp_q_per_level', 'first_div_level_hist', 'block_rel_drift']:
            print(f'{f:22s}', v.get(f))
print('\n=== B. Ternary tarama (ilk 256 test), çözülen% ===')
print(pd.DataFrame([{'ad': k[2:], 'katman': v['n_layers'], 'rel_err': round(v['rel_err'], 3), 'çözülen%': v['solved_pct'],
                     'hücre%': v['cell_pct'], 'FP_ile_aynı_kod%': v['same_codes_as_fp_pct']}
                    for k, v in R.items() if k.startswith('B/')]).to_string(index=False))

## 12. Seed tekrarı — E3-uzun, seed 1 ve 2 (her biri ≈55 dk, A100)

**Gerekenler:** 1–4. bölümler, ardından 6. bölüm (`BATCH` ve `MICRO` değerlerini ayarlar). 5 ve 7–11 atlanabilir.
E3-uzun (seed 0, %71.60) ile birebir aynı tarif; değişen yalnızca veri sırası ve bozma için kullanılan seed.
FP checkpoint'ten başlandığı için başlangıç ağırlıkları her seed'de aynı.
Oturum koparsa hücreyi tekrar çalıştır: biten seed atlanır, yarım kalan checkpoint'ten devam eder.

In [ ]:
for SEED in (1, 2):
    res_path = f'{QAT_DIR}/E3_ternary_ft_5k_s{SEED}_result.json'
    if os.path.exists(res_path):
        print(f'seed {SEED} zaten bitmiş:', json.load(open(res_path))['test']['solved_pct']); continue
    torch.manual_seed(SEED)
    _, tk, net = Q.load_official(**NCM)
    Q.ternarize_linears(net, 'body')
    cfg_s = Q.TrainConfig(run_name=f'E3_ternary_ft_5k_s{SEED}', seed=SEED, steps=5000, batch_size=BATCH,
                          micro_batch=MICRO, peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500)
    Q.train(net, tk, train_seqs, holdout_seqs, cfg_s, QAT_DIR, test_items, test_refs)
    r_s = Q.evaluate_test(net, tk, test_items, test_refs)
    print(f"E3-uzun seed {SEED} tam test: %{r_s['solved_pct']:.2f}   (seed 0: %71.60)")
    json.dump({'test': r_s, 'cfg': cfg_s.__dict__}, open(res_path, 'w'), indent=1)
    del net; torch.cuda.empty_cache()

## 13. E3-B0FP — blok 0 FP, kalan 28 gövde katmanı ternary (≈55 dk, A100)

**Gerekenler:** 1–4. bölümler. E3-uzun (seed 0) tarifi; tek fark blok 0'ın 4 doğrusal katmanının FP kalması
(PTQ teşhisinde en hassas blok). Sonuç: %71.30 (3 seed ortalaması 71.15 ± 0.40 ile fark gürültü içinde). Kayıtlı koşu 219 dk sürdü (GPU türü kaydedilmedi).

In [ ]:
import os, json, torch, torch.nn as nn
if 'BATCH' not in globals():
    BATCH = 256
    _gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    MICRO = 256 if _gb >= 35 else (128 if _gb >= 20 else 64)

RUN = 'E3_ternary_ft_5k_b0fp'
res_path = f'{QAT_DIR}/{RUN}_result.json'
if os.path.exists(res_path):
    print('Zaten bitmiş:', json.load(open(res_path))['test'])
else:
    torch.manual_seed(0)
    _, tk, net = Q.load_official(**NCM)
    Q.ternarize_linears(net, 'body')
    # blok 0'daki TernaryLinear'ları aynı ağırlıklarla düz nn.Linear'a geri çevir
    kept = []
    for name, m in list(net.named_modules()):
        if isinstance(m, Q.TernaryLinear) and '.blocks.0.' in f'.{name}.':
            parent_name, _, attr = name.rpartition('.')
            parent = net.get_submodule(parent_name)
            lin = nn.Linear(m.in_features, m.out_features, bias=m.bias is not None,
                            device=m.weight.device, dtype=m.weight.dtype)
            with torch.no_grad():
                lin.weight.copy_(m.weight)
                if m.bias is not None:
                    lin.bias.copy_(m.bias)
            setattr(parent, attr, lin); kept.append(name)
    n_tern = sum(isinstance(m, Q.TernaryLinear) for m in net.modules())
    print('FP bırakılan:', kept)
    print('Ternary katman sayısı:', n_tern, '(beklenen 28)')
    assert len(kept) == 4 and n_tern == 28

    cfg_b = Q.TrainConfig(run_name=RUN, seed=0, steps=5000, batch_size=BATCH, micro_batch=MICRO,
                          peak_lr=1e-4, floor_lr=1e-5, warmup=100, eval_every=1000, ckpt_every=500)
    Q.train(net, tk, train_seqs, holdout_seqs, cfg_b, QAT_DIR, test_items, test_refs)
    r_b = Q.evaluate_test(net, tk, test_items, test_refs)
    json.dump({'test': r_b, 'cfg': cfg_b.__dict__, 'fp_layers': kept}, open(res_path, 'w'), indent=1)
    print('E3-B0FP tam test:', r_b)

## 10. Özet

In [ ]:
import pandas as pd
rows = []
def add(name, path, key='test'):
    if os.path.exists(path):
        r = json.load(open(path)); r = r.get(key, r)
        rows.append({'deney': name, **{k: round(v, 2) for k, v in r.items() if k != 'n'}, 'n': r['n']})
add('Resmî FP (referans)', f'{QAT_DIR}/official_fp_test.json', key=None)
add('E3 ternary, FP\'den ince ayar', f'{QAT_DIR}/E3_result.json')
add('E1 FP, baştan (kısa bütçe)', f'{QAT_DIR}/E1_result.json')
add('E2 ternary, baştan (kısa bütçe)', f'{QAT_DIR}/E2_result.json')
add('E1 FP, baştan (uzun bütçe)', f'{QAT_DIR}/E1_long_result.json')
add('E2 ternary, baştan (uzun bütçe)', f'{QAT_DIR}/E2_long_result.json')
add('E1 FP, baştan, LM1B tarzı bozma (uzun)', f'{QAT_DIR}/E1_lm1bcorr_result.json')
display(pd.DataFrame(rows))

## Nasıl yorumlanır

- **Kontrol 1 ≈ 1.0 değilse** eğitim sonuçlarına güvenme: girdi kurucumuz resmî üretimle uyuşmuyor demektir.
- **E3 yüksekse** (örneğin %86'ya yakın): ternary NCM mümkün ve ucuz; FP modelden kısa bir ince ayarla
  ulaşılabiliyor. Bu, PTQ'nun %0'ına karşı çok güçlü bir sonuç.
- **E1 düşükse** (kısa bütçede %86'nın çok altında): ya bütçe yetersiz ya da varsayımlarımız (bozma oranı, α)
  makaledekinden farklı. Bu durumda E1 ile E2'yi birbirleriyle karşılaştır, resmî %86 ile değil.
- **E2 / E1 oranı** asıl sonuçtur: aynı bütçede ternary, FP'nin ne kadarına ulaşıyor?
- Tek seed ve kısaltılmış bütçe: sonuç umut vericiyse aynı deneyleri 2–3 seed ve daha uzun bütçeyle tekrarla.